# MannuMitra — Deep Learning Disease Detection Module 04
## MobileNetV2 Plant Leaf Disease Classifier Suite & Evaluation

**Architecture:** MobileNetV2 Transfer Learning (Pre-trained on ImageNet, fine-tuned on Karnataka agricultural crop disease datasets).
**Supported Crops:**
1. **Arecanut (9 classes):** Healthy, Koleroga (Fruit Rot), Yellow Leaf Disease, Bud Rot, Stem Bleeding, Inflorescence Dieback, Nut Splitting, Algal Spot, Sun Scorch
2. **Cotton (4 classes):** Bacterial Blight, Curl Virus, Fusarium Wilt, Healthy
3. **Maize (4 classes):** Common Rust, Gray Leaf Spot, Northern Corn Leaf Blight, Healthy
4. **Rice (5 classes):** Bacterial Leaf Blight, Brown Spot, Leaf Blast, Sheath Blight, Healthy
5. **Tomato (10 classes):** Bacterial Spot, Early Blight, Late Blight, Leaf Mold, Septoria Leaf Spot, Spider Mites, Target Spot, Yellow Leaf Curl Virus, Mosaic Virus, Healthy
6. **Crop Validator CNN (5 classes):** Identifies leaf crop type (Arecanut, Cotton, Maize, Rice, Tomato) before routing to the specialist classifier.


In [ ]:
import os
import json
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Ensure Keras uses PyTorch backend
os.environ['KERAS_BACKEND'] = 'torch'
import keras
print('Keras version:', keras.__version__)

ROOT_DIR = Path('..')
MODELS_DIR = ROOT_DIR / 'models' / 'disease_detection'
SAMPLES_DIR = ROOT_DIR / 'demo_samples' / 'disease_images'

print('Models directory:', MODELS_DIR)
print('Samples directory:', SAMPLES_DIR)


### 1. Load All Trained MobileNetV2 Disease Models
Verify model weights, input dimensions $(224 \times 224 \times 3)$, and output class dimensions.

In [ ]:
crops = ['crop_validator', 'arecanut', 'cotton', 'maize', 'rice', 'tomato']
models = {}
class_mappings = {}
disease_infos = {}

for crop in crops:
    crop_dir = MODELS_DIR / crop
    model_path = crop_dir / 'model.keras'
    names_path = crop_dir / 'class_names.json'
    info_path = crop_dir / 'disease_info.json'
    
    assert model_path.exists(), f'Missing model: {model_path}'
    m = keras.models.load_model(str(model_path))
    models[crop] = m
    
    if names_path.exists():
        with open(names_path, 'r', encoding='utf-8') as f:
            class_mappings[crop] = json.load(f)
            
    if info_path.exists():
        with open(info_path, 'r', encoding='utf-8') as f:
            disease_infos[crop] = json.load(f)
            
    print(f'[{crop.upper()}] Input shape: {m.input_shape}, Output classes: {m.output_shape[-1]}')


### 2. Live Field Inference Demonstration on Demo Sample Images
Run inference on real field demo images, preprocess to $(224, 224)$, scale to $[0, 1]$, and display top predictions.

In [ ]:
demo_images = [
    ('arecanut', SAMPLES_DIR / 'arecanut_koleroga_fruit_rot.jpg'),
    ('cotton', SAMPLES_DIR / 'cotton_bacterial_blight.jpg'),
    ('maize', SAMPLES_DIR / 'maize_common_rust.jpg'),
    ('rice', SAMPLES_DIR / 'rice_brown_spot.jpg'),
    ('tomato', SAMPLES_DIR / 'tomato_early_blight.jpg')
]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for idx, (crop, img_path) in enumerate(demo_images):
    ax = axes[idx]
    if img_path.exists():
        img = Image.open(img_path).convert('RGB')
        img_resized = img.resize((224, 224))
        arr = np.array(img_resized, dtype=np.float32) / 255.0
        arr_batch = np.expand_dims(arr, axis=0)
        
        preds = models[crop].predict(arr_batch, verbose=0)[0]
        top_idx = int(np.argmax(preds))
        conf = float(preds[top_idx]) * 100.0
        
        # Resolve class name
        c_map = class_mappings.get(crop, {})
        idx_to_name = {v: k for k, v in c_map.items()} if isinstance(list(c_map.values())[0] if c_map else 0, int) else c_map
        class_name = idx_to_name.get(top_idx, idx_to_name.get(str(top_idx), f'Class {top_idx}'))
        
        ax.imshow(img)
        ax.set_title(f'{crop.capitalize()}\nPred: {class_name}\nConf: {conf:.1f}%', fontsize=10)
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, f'{img_path.name}\nNot Found', ha='center', va='center')
        ax.axis('off')

plt.tight_layout()
plt.show()


### 3. Integrated Curative Agronomic Advice
Display bilingual curative treatment and chemical recommendations for the identified disease.

In [ ]:
sample_disease = 'koleroga_fruit_rot'
info = disease_infos.get('arecanut', {}).get(sample_disease, {})
print('Disease:', info.get('name_en', 'Koleroga / Mahali'))
print('Kannada Name:', info.get('name_kn', 'ಕೊಳೆರೋಗ (ಮಹಾಲಿ)'))
print('Symptoms:', info.get('symptoms_en', 'Dark green water-soaked lesions on nuts...'))
print('Curative Treatment (Chemical):', info.get('chemical_cure_en', 'Spray 1% Bordeaux mixture before monsoon'))
print('Organic Management:', info.get('organic_cure_en', 'Ensure clean drainage; remove diseased fallen nuts'))
